In [1]:
def updateVelocity(v_b, v_w, A, CdRho, m_b, dt):
    v_rel = v_b - v_w
    F = ((CdRho * A * v_rel**2) * np.sign(v_rel)) / m_b
    return v_b - (F * dt)

def pointDistance(p1, p2):
    return np.sqrt(np.sum((np.array(p1) - np.array(p2))**2))

In [27]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from matplotlib import pyplot as plt

# Pre-defined simulation grids
speeds = np.linspace(0, 8, 21)
angles = np.linspace(0, 180, 91)

# Widget controls 
v_wx_slider = widgets.FloatSlider(value=-3.0, min=-5.0, max=5.0, step=0.5, description='v_w X (m/s)')
r_b_slider = widgets.FloatSlider(value=0.05, min=0.01, max=0.15, step=0.005, readout_format='.3f', description='Radius r_b')
CdRho_slider = widgets.FloatSlider(value=0.47 * 375, min=50, max=300, step=10, description='Cd * Rho')
m_b_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Mass m_b')
submersion_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Submersion') 
target_x_slider = widgets.FloatSlider(value=0.6, min=-2.0, max=2.0, step=0.01, description='Target X')
target_y_slider = widgets.FloatSlider(value=1.4, min=0.5, max=3.0, step=0.01, description='Target Y')

ui = widgets.VBox([
    widgets.HBox([v_wx_slider]),
    widgets.HBox([r_b_slider, CdRho_slider, m_b_slider, submersion_slider]),
    widgets.HBox([target_x_slider, target_y_slider])
])

out = widgets.Output()


def update_plot(change):
    with out:
        clear_output(wait=True)

        # Current parameters from sliders
        water_speed = np.array([v_wx_slider.value, 0.0])
        target = [target_x_slider.value, target_y_slider.value]

        custom_constants = {
            'CdWater': CdRho_slider.value / 375.0,
            'rhoWater': 375.0,
            'massBall': m_b_slider.value,
            'radiusBall': r_b_slider.value,
            'radius': 0.0875,
            'submersion': submersion_slider.value, 
            'dt': 1.0 / 72
        }

        errors = np.zeros((len(speeds), len(angles)))
        for v_idx, speed in enumerate(speeds):
            for a_idx, angle in enumerate(angles):
                # Set up constants
                CdWater = custom_constants['CdWater']
                rhoWater = custom_constants['rhoWater']
                massBall = custom_constants['massBall']
                radius = custom_constants['radius']
                submersion = custom_constants['submersion']
                dt = custom_constants['dt']
                
                areaBall = submersion * np.pi * (r_b_slider.value**2)

                # Run  trajectory simulation
                angle_rad = np.radians(angle)
                velocity = np.array([speed * np.cos(angle_rad), speed * np.sin(angle_rad)])
                position = np.array([0.0, 0.0])
                prevdist = pointDistance(target, position)
                trajectory = np.reshape(np.concatenate((position.copy(), np.array([prevdist]))), (1,3))
                ongoing = True

                while ongoing:
                    velocity = updateVelocity(v_b=velocity, v_w=water_speed, A=areaBall, CdRho=CdWater*rhoWater, m_b=massBall, dt=dt)
                    position += velocity * dt
                    dist = pointDistance(position, target)
                    newrow = np.reshape(np.concatenate((position.copy(), np.array([dist]))), (1,3))
                    trajectory = np.concatenate((trajectory, newrow))
                    if position[1] > target[1] and dist > prevdist:
                        ongoing = False
                    if np.sign(water_speed[0])*position[0] > 3:
                        ongoing = False
                    prevdist = dist

                errors[v_idx, a_idx] = np.abs(np.min(trajectory[:,2]) - radius)

        my_errors = errors - 0.1
        my_errors = np.maximum(my_errors, np.zeros_like(errors))

        min_x, min_y, max_x, max_y = [], [], [], []
        successful_angles = []
        successful_speeds = []

        for angle_idx in range(len(angles)):
            angle = angles[angle_idx]
            success_mask = my_errors[:, angle_idx] < 1e-6
            if any(success_mask):
                minspeed = speeds[int(np.min(success_mask.nonzero()[0]))]
                maxspeed = speeds[int(np.max(success_mask.nonzero()[0]))]
                min_x.append(angle)
                min_y.append(minspeed)
                max_x.append(angle)
                max_y.append(maxspeed)
                successful_angles.append(angle)
                successful_speeds.extend(speeds[success_mask])

        # Plotting
        plt.figure(figsize=(7, 4.5))
        plt.xlim(0, 180)
        plt.ylim(0, 8)
        plt.imshow(my_errors, cmap='viridis', origin='lower',
                   extent=[angles.min(), angles.max(), speeds.min(), speeds.max()],
                   aspect='auto')
        plt.plot(max_x, max_y, color='red', label='Upper Bound' if max_x else "")
        plt.plot(min_x, min_y, color='red', label='Lower Bound' if min_x else "")
        plt.colorbar(label='error (m)')
        plt.xticks([0, 45, 90, 135, 180])

        target_angle = (np.arctan2(target[1], target[0]) / np.pi) * 180
        plt.plot([target_angle, target_angle], [0, 8], color='orange', linestyle='dashed', label='Direct Target Angle')
        plt.title(f'Interactive Solution Space\nTarget: {target}, Water v_w: {water_speed[0]} m/s')
        plt.xlabel('Launch Angle (degrees)')
        plt.ylabel('Launch Speed (m/s)')
        plt.tight_layout()
        plt.show()

        # Calculate and print metrics
        num_solutions = np.sum(my_errors < 1e-6)
        if num_solutions > 0:
            min_required_speed = np.min(successful_speeds)
            min_angle = np.min(successful_angles)
            max_angle = np.max(successful_angles)
            print(f"--- Solutions Telemetry ---")
            print(f"Number of successful solutions: {num_solutions} (on current grid)")
            print(f"Minimum required launch speed: {min_required_speed:.2f} m/s")
            print(f"Range of successful angles: {min_angle:.1f}° to {max_angle:.1f}°")
        else:
            print(f"--- Solutions Telemetry ---")
            print("No successful solutions found for current parameters.")

for widget in [v_wx_slider, r_b_slider, CdRho_slider, m_b_slider, submersion_slider, target_x_slider, target_y_slider]:
    widget.observe(update_plot, names='value')

# Initialize and display
display(ui, out)
update_plot(None)

Output()